<a href="https://colab.research.google.com/github/jd4068/Hydra_Connectome/blob/main/code_main_paper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Overview of computational framework

The analysis pipeline converts single-cell transcriptomic data into predicted connectivity patterns between individual cells and cell types by integrating gene expression with ligand–receptor interaction predictions. Single-cell RNA-sequencing output was represented as a gene-by-cell expression matrix and processed to obtain cell-type–level expression profiles through normalization, transformation, and aggregation by annotated cell identities.

Predicted binding potentials between G protein–coupled receptors (GPCRs) and candidate neuropeptides were used to weight interactions between expressing cells. Receptor and peptide expression matrices were combined with the interaction prediction matrix to generate directed connectivity matrices describing potential signaling from peptide-expressing cells to receptor-expressing cells. These matrices were further aggregated to the cell-type level to identify higher-order communication patterns and signaling hubs.

Although this study focuses on GPCR–neuropeptide interactions, the framework is modular and can be readily adapted to incorporate alternative interaction datasets, including experimentally validated protein–protein interactions, receptor–ligand databases, or other molecular interaction predictions of interest.

The analysis requires two primary inputs: (i) a matrix of single-cell transcriptomic data with annotated cell-type identities, and (ii) a matrix of predicted interactions between proteins of interest, linked to the same gene or protein identifiers used in the transcriptomic dataset. Consistent naming between expression data and interaction predictions is essential to enable correct integration.

Because the datasets analyzed here are large, intermediate representations were converted into multiple matrix formats, including sparse and dense NumPy arrays, and memory-mapped datasets to enable efficient storage and computation. These additional preprocessing steps were implemented to accommodate memory and performance constraints. When working with smaller datasets, many of these intermediate conversions and optimizations can be omitted without altering the overall analysis workflow.

In [1]:
!pip install openpyxl biopython

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.6 MB/s eta 0:00:00


In [2]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#the analyisis of receptor and ligand expression
cellNamestoTypes = pd.read_csv('/content/drive/MyDrive/Yuste_Paper_Figures/hydra_neuron_cell_type.csv')
expression_matrix = pd.read_csv('/content/drive/MyDrive/Yuste_Paper_Figures/hydra_neuron_expression.csv')
# read in connections_table <- read.csv("/Users/johanna/Hydra/rnaSeq/scores_hydra_ALLPeptides.csv", sep = ';')


In [ ]:

# ---- 1. Load data ----
# CSV
pair_csv = pd.read_csv(
    '/content/drive/MyDrive/Yuste_Paper_Figures/scores_hydra_ALLPeptides.csv',
    sep=';'
)

# Excel
pair_xlsx = pd.read_excel(
    '/content/drive/MyDrive/Yuste_Paper_Figures/scores_hydra_ALL.xlsx'
)

# ---- 2. Clean peptide columns ----
# CSV file: remove everything before last dot
pair_csv['peptide'] = pair_csv['peptide'].str.replace(r'.*\.', '', regex=True)

# XLSX file: remove everything before 'HVAEP1-' (your rule)
pair_xlsx['peptide'] = pair_xlsx['peptide'].str.replace(r'.*HVAEP1-', '', regex=True)

# ---- 3. Merge both tables vertically ----
pair_table_conc = pd.concat([pair_csv, pair_xlsx], ignore_index=True)

# ---- 4. Extract peptide_TG = part before first '_' ----
pair_table_conc['peptide_TG'] = pair_table_conc['peptide'].str.split('_').str[0]

# ---- 5. Sort by peptide_TG ----
pair_table_conc = pair_table_conc.sort_values(by='peptide_TG')

# ---- 6. Ensure iptm numeric ----
pair_table_conc['iptm'] = pd.to_numeric(pair_table_conc['iptm'], errors='coerce')

# ---- 7. Extract top 10% rows by iptm ----
pair_table_conc_top10 = pair_table_conc.nlargest(
    int(len(pair_table_conc) * 0.10),
    'iptm'
)

# ---- Optional: print summary ----
print("Merged rows:", len(pair_table_conc))
print("Top 10% rows:", len(pair_table_conc_top10))


In [ ]:
#if you want to try a subset to see how it looks
#find all those that have value HVAEP9.G017228_12 in peptide+
#HVAEP9_G017228_12 = pair_table_conc[pair_table_conc['peptide'].str.contains('GLWamide1_HVAEP1-G018128')]
#HVAEP9_G017228_12 = HVAEP9_G017228_12.nlargest(int(len(HVAEP9_G017228_12)*0.1), 'iptm')

#sort by iptm
#HVAEP9_G017228_12 = HVAEP9_G017228_12.sort_values(by=['iptm'])
#find the receptor column values in the expression table
#HVAEP9_G017228_12_receptor = HVAEP9_G017228_12['receptor']

In [ ]:
# ------------------------------
# 1. Prepare expression matrix
# ------------------------------
exp_mat_backup = expression_matrix.copy()

# Make first column into index
expression_matrix.index = expression_matrix.iloc[:, 0]
expression_matrix = expression_matrix.drop(expression_matrix.columns[0], axis=1)

# Remove everything before the first "-" including the "-"
expression_matrix.index = expression_matrix.index.str.split('-').str[1]

# ------------------------------
# 2. Subset based on receptor list
# ------------------------------
subset = expression_matrix.loc[
    expression_matrix.index.isin(HVAEP9_G017228_12_top10_receptor)
]

# Ensure same order as receptor list
subset = subset.reindex(HVAEP9_G017228_12_top10_receptor)

# ------------------------------
# 3. Create proper score vector (10 → 1)
# ------------------------------
num_rows = len(subset)
scores = list(range(10, 10 - num_rows, -1))   # e.g. [10,9,8,...]

# If fewer than 10 rows:
if len(scores) < num_rows:
    scores += [1] * (num_rows - len(scores))

score_df = pd.DataFrame(scores, index=subset.index, columns=['Score'])

# ------------------------------
# 4. Multiply each receptor row by score
# ------------------------------
weighted_df = subset.mul(score_df['Score'], axis=0)

# ------------------------------
# 5. Replace column names with cell types
# ------------------------------
weighted_df.columns = cellNamestoTypes['cell_type']

# ------------------------------
# 6. Group by cell type
# ------------------------------
weighted_df = weighted_df.groupby(level=0, axis=1).sum()

# ------------------------------
# 7. Prepare for plotting
# ------------------------------
import matplotlib.pyplot as plt

data = weighted_df.T.iloc[::-1]   # transpose + invert order
df = pd.DataFrame(data)

# ------------------------------
# 8. Make stacked bar plot
# ------------------------------
plt.figure(figsize=(14, 8))

ax = df.plot(
    kind='bar',
    stacked=True,
    figsize=(14, 8),
    width=0.8,
    colormap=plt.get_cmap("coolwarm")
)

plt.xlabel('Cell Types', fontsize=18)
plt.ylabel('Weighted Expression', fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=14)

# Legend in reversed order
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1],
          title='Receptors',
          bbox_to_anchor=(1.05, 1),
          loc='upper left',
          fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------
# 0. PREP: Fix peptide/receptor naming
# ------------------------------------------------------------

pair_table_conc['peptide_TG']  = 'HVAEP1-' + pair_table_conc['peptide']
pair_table_conc['receptor_TG'] = 'HVAEP1-' + pair_table_conc['receptor']

pair_table_conc_top10['peptide_TG']  = 'HVAEP1-' + pair_table_conc_top10['peptide']
pair_table_conc_top10['receptor_TG'] = 'HVAEP1-' + pair_table_conc_top10['receptor']

# Unique lists
receptors_all   = pair_table_conc['receptor'].drop_duplicates()
receptors_top10 = pair_table_conc_top10['receptor'].drop_duplicates()

peptides_all    = pair_table_conc['peptide_TG'].drop_duplicates()
peptides_top10  = pair_table_conc_top10['peptide_TG'].drop_duplicates()

# ------------------------------------------------------------
# 1. CLEAN cell-type mapping
# ------------------------------------------------------------

cellNamestoTypes['cell'] = cellNamestoTypes['cell'].str.replace('-', '.', regex=False)
cell_names_to_types_dict = dict(zip(cellNamestoTypes['cell'], cellNamestoTypes['cell_type']))

# Apply cell-type names
expression_matrix_types = expression_matrix.copy()
expression_matrix_types = np.log2(1 + expression_matrix_types)
expression_matrix_types.columns = expression_matrix_types.columns.map(cell_names_to_types_dict)

# ------------------------------------------------------------
# 2. REMOVE receptors that are missing in matrix
# ------------------------------------------------------------

# Matrix grouped by cell type
expression_matrix_grouped = expression_matrix_types.groupby(level=0, axis=1).mean()

# Receptors missing entirely
valid_receptors = expression_matrix_grouped.index

missing_receptors = receptors_all[~receptors_all.isin(valid_receptors)]
receptors_all     = receptors_all[~receptors_all.isin(missing_receptors)]
receptors_top10   = receptors_top10[~receptors_top10.isin(missing_receptors)]

# Remove a specific unwanted receptor
receptors_all = receptors_all[~receptors_all.str.contains('G028818', na=False)]

# ------------------------------------------------------------
# 3. Remove bad peptides
# ------------------------------------------------------------

peptides_all    = peptides_all[~peptides_all.str.contains('G018620', na=False)]
peptides_top10  = peptides_top10[~peptides_top10.str.contains('G018620', na=False)]

# ------------------------------------------------------------
# 4. Group expression matrix correctly
# ------------------------------------------------------------

expression_matrix_grouped = expression_matrix_types.groupby(level=0, axis=1).mean()
expression_matrix_grouped = expression_matrix_grouped.drop(columns=['td3'], errors='ignore')
expression_matrix_grouped = expression_matrix_grouped.drop_duplicates()

# ------------------------------------------------------------
# 5. Normalize receptor + peptide names (remove prefix)
# ------------------------------------------------------------

receptors_top10 = receptors_top10.dropna().astype(str).str.split('-').str[1]
peptides_top10  = peptides_top10.reset_index(drop=True).astype(str).str.split('-').str[1]

peptides_all = peptides_all.astype(str).str.split('-').str[1]

# ------------------------------------------------------------
# 6. Subset expression matrices
# ------------------------------------------------------------

expression_matrix_grouped_receptors       = expression_matrix_grouped.loc[receptors_all]
expression_matrix_grouped_receptors_top10 = expression_matrix_grouped.loc[receptors_top10]

expression_matrix_grouped_peptides        = expression_matrix_grouped.loc[peptides_all]
expression_matrix_grouped_peptides_top10  = expression_matrix_grouped.loc[peptides_top10]




In [ ]:
peptides_top10.str.split('-')

In [ ]:
peptides_top10
#make the first col index
peptides_top10 = peptides_top10.reset_index(drop=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a figure with two subplots
#fig, axs = plt.subplots(1, 2, figsize=(12, 3))

# Plot the first clustermap on the left subplot
clustermap = sns.clustermap(expression_matrix_grouped_peptides, cmap="coolwarm")
clustermap.ax_heatmap.set_xlabel('Cell Types', fontsize=12)
clustermap.ax_heatmap.set_ylabel('NeuroPeptide Gene Names', fontsize=12)



# Plot the second clustermap on the right subplot
clustermap = sns.clustermap(expression_matrix_grouped_receptors, cmap="coolwarm")
clustermap.ax_heatmap.set_xlabel('Cell Types', fontsize=12)
clustermap.ax_heatmap.set_ylabel('GPCR Gene Names', fontsize=12)
# Adjust layout



\now its time for some matrix multiplication




In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------
# 1. ALL PEPTIDES
# ------------------------------------------------------------
g1 = sns.clustermap(
    expression_matrix_grouped_peptides,
    cmap="coolwarm",
    figsize=(8, 10)
)
g1.ax_heatmap.set_xlabel('Cell Types', fontsize=12)
g1.ax_heatmap.set_ylabel('Neuropeptide Gene Names', fontsize=12)
plt.show()

# ------------------------------------------------------------
# 2. ALL RECEPTORS
# ------------------------------------------------------------
g2 = sns.clustermap(
    expression_matrix_grouped_receptors,
    cmap="coolwarm",
    figsize=(8, 10)
)
g2.ax_heatmap.set_xlabel('Cell Types', fontsize=12)
g2.ax_heatmap.set_ylabel('GPCR Gene Names', fontsize=12)
plt.show()

# ------------------------------------------------------------
# 3. TOP10 PEPTIDES
# ------------------------------------------------------------
g3 = sns.clustermap(
    expression_matrix_grouped_peptides_top10,
    cmap="coolwarm",
    figsize=(8, 10)
)
g3.ax_heatmap.set_xlabel('Cell Types', fontsize=12)
g3.ax_heatmap.set_ylabel('Top10 Neuropeptide Gene Names', fontsize=12)
plt.show()

# ------------------------------------------------------------
# 4. TOP10 RECEPTORS
# ------------------------------------------------------------
g4 = sns.clustermap(
    expression_matrix_grouped_receptors_top10,
    cmap="coolwarm",
    figsize=(8, 10)
)
g4.ax_heatmap.set_xlabel('Cell Types', fontsize=12)
g4.ax_heatmap.set_ylabel('Top10 GPCR Gene Names', fontsize=12)
plt.show()


In [ ]:
#load data parquet
import pyarrow.parquet as pq
receptor_peptide_df = pd.read_parquet('/content/drive/MyDrive/Yuste_Paper_Figures/receptor_peptide.parquet')
receptor_expression_df = pq.read_table('/content/drive/MyDrive/Yuste_Paper_Figures/expression_matrix_receptors.parquet')
peptide_expression_df = pq.read_table('/content/drive/MyDrive/Yuste_Paper_Figures/expression_matrix_peptides_filtered.parquet')



In [ ]:
peptides_all

In [ ]:
#arrays
receptor_expression = receptor_expression_df.to_pandas().to_numpy().T
receptor_peptide = receptor_peptide_df.to_numpy().T
peptide_expression = peptide_expression_df.to_pandas().to_numpy()

In [ ]:
mask_list = np.zeros((13, 13, 26713))



for i in range(len(peptide_expression)):

    mask_list[i,i] = peptide_expression[i]


In [ ]:
threeD_matrix = np.memmap('threeD_matrix.np', mode='w+', shape=(13,  26713, 26713))

In [ ]:
#load it in from drive numpy object saved
threeD_matrix[:] = np.load('/content/drive/MyDrive/Yuste_Paper_Figures/threeD_matrix.npy')

In [ ]:
def adjacency_matrix(receptor_expression, receptor_peptide, mask):
  return np.dot(np.dot(receptor_expression, receptor_peptide), mask)

#threeD_matrix = np.empty((13,  26713, 26713))
for i, mask in enumerate(mask_list):
  threeD_matrix[i] = adjacency_matrix(receptor_expression, receptor_peptide, mask)



In [ ]:
adj1 = np.memmap('adj1.np', mode='w+', shape=(26713, 26713))
adj1[:] = np.sum(threeD_matrix, axis=0)

In [ ]:
cellnames = pd.read_csv('/content/drive/MyDrive/Yuste_Paper_Figures/cell_names.txt', header=None)
cellNamestoTypes = pd.read_csv('/content/drive/MyDrive/Yuste_Paper_Figures/hydra_neuron_cell_type.csv')
# Create cell type dictionary
cell_names_to_types_dict = dict(zip(cellNamestoTypes['cell'], cellNamestoTypes['cell_type']))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming threeD_matrix, cellnames, cell_names_to_types_dict, and peptides_all are defined
# For example:
# threeD_matrix = np.random.rand(16, 10, 10)
# cellnames = [f'cell{i}' for i in range(10)]
# cell_names_to_types_dict = {f'cell{i}': f'type{i}' for i in range(10)}
# peptides_all = pd.Series([f'peptide{i}' for i in range(16)])

# Set up a 4x4 grid for the subplots
fig, axes = plt.subplots(4, 4, figsize=(20, 20))

# Flatten the axes array for easy iteration
axes = axes.flatten()

for z in range(threeD_matrix.shape[0]):
    mat_2d = threeD_matrix[z, :, :].reshape(-1, threeD_matrix.shape[1])

    # Convert to DataFrame
    mat = pd.DataFrame(mat_2d, index=cellnames, columns=cellnames)
    mat.columns = [str(x).replace('.', '-').replace(',', '').replace("'", "").strip('()') for x in mat.columns]
    mat.columns = [cell_names_to_types_dict[str(x)] for x in mat.columns]

    # Do the same for the index
    mat.index = [str(x).replace('.', '-').replace(',', '').replace("'", "").strip('()') for x in mat.index]
    mat.index = [cell_names_to_types_dict[str(x)] for x in mat.index]

    # Group by the unique names of rows and columns and sample 10% of each group
    mat = mat.groupby(axis=0, level=0).apply(lambda x: x.sample(frac=0.1, random_state=1))
    mat = mat.groupby(axis=1, level=0).apply(lambda x: x.sample(frac=0.1, axis=1, random_state=1))

    # Sort the matrix
    mat = mat.sort_index().sort_index(axis=1)
    mat.reset_index(drop=True, level=0, inplace=True)
    mat.columns = mat.columns.droplevel(0)

    # Apply log transformation and normalization
    log_matrix = np.log2(1 + mat)
    log_matrix.replace([np.inf, -np.inf], 0, inplace=True)
    norm_matrix = log_matrix

    # Drop 'td3' rows and columns
    norm_matrix.drop('td3', axis=0, errors='ignore', inplace=True)
    norm_matrix.drop('td3', axis=1, errors='ignore', inplace=True)

    # Plot the heatmap in the corresponding subplot
    sns.heatmap(data=norm_matrix, cmap='coolwarm', cbar_kws={'label': 'Log2 Intensity'}, ax=axes[z])

    # Add title and labels
    axes[z].set_title('Heatmap of Matrix Connections through ' + peptides_all.iloc[z])
    axes[z].set_xlabel('Receiving')
    axes[z].set_ylabel('Sending')

# Adjust layout
plt.tight_layout()

# Show the figure
plt.show()



In [ ]:

#adj as csv from dirve
adj1 = pd.read_csv('/content/drive/MyDrive/Yuste_Paper_Figures/adj1.csv', index_col=0)

In [ ]:
adj1= pd.DataFrame(adj1, index=cellnames, columns=cellnames)
adj1=adj1

In [ ]:
cellnames = pd.read_csv('/content/drive/MyDrive/Yuste_Paper_Figures/cell_names.txt', header=None)
cellNamestoTypes = pd.read_csv('/content/drive/MyDrive/Yuste_Paper_Figures/hydra_neuron_cell_type.csv')
# Create cell type dictionary
cell_names_to_types_dict = dict(zip(cellNamestoTypes['cell'], cellNamestoTypes['cell_type']))

In [ ]:

# Ensure the column names are strings before applying replace
#adj1_df_types = adj1_df.copy()

# Check the type of each column name and convert to string if necessary
#adj1_df_types.columns = [str(x).replace('.', '-') for x in adj1_df.columns]
adj1.columns = [str(x).replace('.', '-').replace(',', '').replace("'", "").strip('()') for x in adj1.columns]
# Assuming cell_names_to_types_dict is defined somewhere in your code
# Example: cell_names_to_types_dict = {'name1': 'type1', 'name2': 'type2', ...}
adj1.columns = [cell_names_to_types_dict[str(x)] for x in adj1.columns]

# Do the same for the index
#adj1_df_types.index = [str(x).replace('.', '-') for x in adj1_df.index]
adj1.index = [str(x).replace('.', '-').replace(',', '').replace("'", "").strip('()') for x in adj1.index]
adj1.index = [cell_names_to_types_dict[str(x)] for x in adj1.index]

# Print to verify


In [ ]:
# adj1 is your DataFrame adjacency matrix
adj_matrix = adj1.values

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd

# Assuming 'adj_matrix' is your NumPy array adjacency matrix

# Assuming 'node_labels' is a list of your neuron labels

# Convert to directed NetworkX graph
G = nx.from_numpy_array(adj_matrix, create_using=nx.DiGraph)

In [ ]:
node_labels = list(adj1.columns)
G = nx.relabel_nodes(G, {i: label for i, label in enumerate(node_labels)})
G.remove_edges_from(nx.selfloop_edges(G)) # Remove self-loops if present

# 1. Calculate total degree (in-degree + out-degree) for each neuron
total_degrees = np.array([G.in_degree(n) + G.out_degree(n) for n in G.nodes()])

# 2. Calculate the average connection count
average_degree = np.mean(total_degrees)
print(f"Average connection count per neuron: {average_degree:.2f}")

# 3. Count how many neurons are connected to more than 50% of other neurons
num_neurons = G.number_of_nodes()
degree_threshold = (num_neurons - 1) * 0.5 # Threshold is 50% of connections to other nodes

highly_connected_neurons_count = np.sum(total_degrees > degree_threshold)
print(f"Number of neurons connected to more than 50% of other neurons: {highly_connected_neurons_count}")

# 4. Make a plot of the degree distribution
plt.figure(figsize=(10, 6))
plt.hist(total_degrees, bins=range(int(max(total_degrees)) + 2), align='left', rwidth=0.8)
plt.axvline(average_degree, color='red', linestyle='dashed', linewidth=1, label=f'Average Degree ({average_degree:.2f})')
plt.axvline(degree_threshold, color='green', linestyle='dashed', linewidth=1, label=f'50% Threshold ({degree_threshold:.2f})')
plt.xlabel('Number of Connections (Total Degree)')
plt.ylabel('Number of Neurons')
plt.title('Distribution of Neuron Connection Counts')
plt.xticks(range(0, int(max(total_degrees)) + 2, max(1, int(max(total_degrees) / 10)))) # Adjust x-ticks
plt.legend()
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm

# --- 1. Graph Setup ---

N = adj_matrix.shape[0]





In [ ]:
#count to how many other nueorns each neuron is conencted give me the average conenction, how may are conencted to more than 50% and make a plot



In [ ]:
# Convert to directed NetworkX graph
G = nx.from_numpy_array(adj_matrix, create_using=nx.DiGraph)
G.remove_edges_from(nx.selfloop_edges(G))



In [ ]:
# --- 2. Vectorized Degree Calculation ---
# Sum rows (out-degree) and columns (in-degree)
out_degrees = np.sum(adj_matrix > 0.05, axis=1)
in_degrees = np.sum(adj_matrix > 0.05, axis=0)
total_degrees = out_degrees + in_degrees

# Define rich nodes as >50% connectivity
degree_threshold = (N - 1) / 2
rich_nodes = np.where(total_degrees > degree_threshold)[0]
print(f"Rich nodes (degree > {degree_threshold:.1f}): {rich_nodes}")

# --- 3. Rich-Club Coefficient (Single Threshold) ---
Nk = len(rich_nodes)
if Nk < 2:
    phi_k = np.nan
else:
    sub_adj = adj_matrix[np.ix_(rich_nodes, rich_nodes)]
    Mk = np.sum(sub_adj > 0)
    max_possible = Nk * (Nk - 1)  # directed, no self-loops
    phi_k = Mk / max_possible

print(f"Rich-club coefficient Φ(k) = {phi_k:.3f}")

In [ ]:
G

In [ ]:
# --- 4. Generate Random Graphs ---
num_random = 1
phi_random = []
for i in tqdm(range(num_random), desc="Randomizing"):
    G_rand = G
    nx.directed_edge_swap(G_rand, nswap=0.5*G_rand.number_of_edges(), max_tries=100)
    adj_rand = nx.to_numpy_array(G_rand)

    out_rand = np.sum(adj_rand > 0, axis=1)
    in_rand = np.sum(adj_rand > 0, axis=0)
    total_rand = out_rand + in_rand

    rich_rand = np.where(total_rand > degree_threshold)[0]
    Nk_rand = len(rich_rand)
    if Nk_rand < 2:
        phi_rand.append(np.nan)
    else:
        sub_adj_rand = adj_rand[np.ix_(rich_rand, rich_rand)]
        Mk_rand = np.sum(sub_adj_rand > 0)
        max_possible_rand = Nk_rand * (Nk_rand - 1)
        phi_rand.append(Mk_rand / max_possible_rand)

phi_random_mean = np.nanmean(phi_rand)
phi_random_std = np.nanstd(phi_rand)
phi_norm = phi_k / phi_random_mean if phi_random_mean else np.nan

print(f"Normalized Φ(k) = {phi_norm:.3f}")
print(f"Random mean Φ = {phi_random_mean:.3f} ± {phi_random_std:.3f}")


In [ ]:

# --- 5. Save Results ---
results = pd.DataFrame({
    'Nk': [Nk],
    'phi_k': [phi_k],
    'phi_random_mean': [phi_random_mean],
    'phi_random_std': [phi_random_std],
    'phi_norm': [phi_norm]
})
results.to_csv('/content/drive/MyDrive/rich_club_summary.csv', index=False)
print("✔ CSV saved to /content/drive/MyDrive/rich_club_summary.csv")

# Save raw random values for diagnostics
pd.Series(phi_random).to_csv('/content/drive/MyDrive/rich_club_random_values.csv', index=False)
print("✔ Random Φ values saved to /content/drive/MyDrive/rich_club_random_values.csv")


In [ ]:


# --- 6. Plot ---
plt.figure(figsize=(6, 4))
plt.bar(['Real', 'Random Mean'], [phi_k, phi_random_mean], yerr=[0, phi_random_std], color=['orange', 'gray'])
plt.axhline(1, color='black', linestyle='--', label='Null Expectation')
plt.ylabel('Rich-club coefficient Φ')
plt.title(f'Rich-Club Coefficient at Degree > {(N-1)/2:.1f}')
plt.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/rich_club_plot.png', dpi=300)
print("✔ Plot saved to /content/drive/MyDrive/rich_club_plot.png")



In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from tqdm import tqdm

def compute_rich_club_coefficient(G, max_k=None):
    """
    Compute Φ(k) for each degree threshold k in a directed graph G.

    Returns:
        phi_k: dict of {k: Φ(k)}
        n_k: dict of {k: number of nodes with degree > k}
    """
    degrees = dict(G.degree())
    max_k = max_k or max(degrees.values())

    phi_k = {}
    n_k = {}
    for k in range(max_k + 1):
        rich_nodes = [n for n, deg in degrees.items() if deg > k]
        Nk = len(rich_nodes)
        if Nk < 2:
            phi_k[k] = np.nan
            n_k[k] = Nk
            continue
        subG = G.subgraph(rich_nodes)
        Mk = subG.number_of_edges()
        max_possible = Nk * (Nk - 1)  # Directed, no self-loops
        phi_k[k] = Mk / max_possible
        n_k[k] = Nk
    return phi_k, n_k

def randomize_graph(G, num_random=100, seed=None):
    """Generate random graphs preserving degree distribution."""
    rng = np.random.default_rng(seed)
    return [
        nx.double_edge_swap(G.copy(), nswap=5*G.number_of_edges(), max_tries=1000)
        for _ in range(num_random)
    ]

def rich_club_analysis(G, num_random=100, seed=42):
    """
    Perform rich-club coefficient analysis with normalization.

    Args:
        G (nx.DiGraph): Directed graph.
        num_random (int): Number of random graphs for null model.

    Returns:
        k_vals: degree thresholds
        phi_real: real Φ(k)
        phi_norm: normalized Φ(k) values
        std_random: std dev of random Φ(k)
    """
    phi_real, _ = compute_rich_club_coefficient(G)
    k_vals = sorted(phi_real.keys())

    # Compute Φrandom(k) for each random graph
    phi_random_all = {k: [] for k in k_vals}
    for G_rand in tqdm(randomize_graph(G, num_random, seed=seed), desc="Randomizing"):
        phi_rand, _ = compute_rich_club_coefficient(G_rand)
        for k in k_vals:
            phi_random_all[k].append(phi_rand.get(k, np.nan))

    phi_random_avg = {k: np.nanmean(phi_random_all[k]) for k in k_vals}
    phi_random_std = {k: np.nanstd(phi_random_all[k]) for k in k_vals}

    # Normalize
    phi_norm = {
        k: phi_real[k] / phi_random_avg[k] if phi_random_avg[k] else np.nan
        for k in k_vals
    }

    return k_vals, phi_real, phi_norm, phi_random_std

def plot_rich_club(k_vals, phi_real, phi_norm, std_random):
    plt.figure(figsize=(10, 5))

    # Plot raw Φ(k)
    plt.subplot(1, 2, 1)
    plt.plot(k_vals, [phi_real[k] for k in k_vals], label='Φ(k)', marker='o')
    plt.xlabel('Degree threshold k')
    plt.ylabel('Rich-club coefficient Φ(k)')
    plt.title('Raw Rich-Club Coefficient')
    plt.grid(True)

    # Plot normalized Φ(k)
    plt.subplot(1, 2, 2)
    norm_vals = [phi_norm[k] for k in k_vals]
    std_vals = [std_random[k] for k in k_vals]
    plt.plot(k_vals, norm_vals, label='Φ_norm(k)', color='blue', marker='o')
    plt.fill_between(k_vals,
                     [v + s for v, s in zip(norm_vals, std_vals)],
                     [v - s for v, s in zip(norm_vals, std_vals)],
                     alpha=0.3, label='±1σ', color='blue')
    plt.axhline(1, color='gray', linestyle='--')
    plt.axhline(1 + np.nanmean(std_vals), color='red', linestyle='--', label='1 + σ')
    plt.xlabel('Degree threshold k')
    plt.ylabel('Normalized Φ(k)')
    plt.title('Normalized Rich-Club Coefficient')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
# Assume adj1 is your adjacency DataFrame
G = nx.from_numpy_array(adj1.values, create_using=nx.DiGraph)
G.remove_edges_from(nx.selfloop_edges(G))  # Remove self-loops

k_vals, phi_real, phi_norm, std_random = rich_club_analysis(G, num_random=100)
plot_rich_club(k_vals, phi_real, phi_norm, std_random)


In [ ]:
import pandas as pd

# === Save results as CSV ===
df_rc = pd.DataFrame({
    'k': k_vals,
    'phi_real': [phi_real[k] for k in k_vals],
    'phi_norm': [phi_norm[k] for k in k_vals],
    'std_random': [std_random[k] for k in k_vals]
})

df_rc.to_csv('/content/drive/MyDrive/rich_club_results.csv', index=False)
print("✔ CSV saved to /content/drive/MyDrive/rich_club_results.csv")

# === Replot and save figure ===
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

# Raw Φ(k)
plt.subplot(1, 2, 1)
plt.plot(k_vals, df_rc['phi_real'], marker='o')
plt.xlabel('Degree threshold k')
plt.ylabel('Rich-club coefficient Φ(k)')
plt.title('Raw Rich-Club Coefficient')
plt.grid(True)

# Normalized Φ(k)
plt.subplot(1, 2, 2)
plt.plot(k_vals, df_rc['phi_norm'], color='blue', marker='o', label='Φ_norm(k)')
plt.fill_between(k_vals,
                 df_rc['phi_norm'] + df_rc['std_random'],
                 df_rc['phi_norm'] - df_rc['std_random'],
                 alpha=0.3, label='±1σ', color='blue')
plt.axhline(1, color='gray', linestyle='--')
plt.axhline(1 + df_rc['std_random'].mean(), color='red', linestyle='--', label='1 + σ')
plt.xlabel('Degree threshold k')
plt.ylabel('Normalized Φ(k)')
plt.title('Normalized Rich-Club Coefficient')
plt.legend()
plt.grid(True)
plt.tight_layout()

# Save the figure
plt.savefig('/content/drive/MyDrive/rich_club_plot.png', dpi=300)
print("✔ Plot saved to /content/drive/MyDrive/rich_club_plot.png")


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

def detect_rich_club(connectivity_matrix, percentile_threshold=80, plot=True):
    """
    Identifies rich club nodes in a neural connectivity matrix.

    Args:
        connectivity_matrix (np.ndarray): Square matrix (N x N) with weights.
        percentile_threshold (float): Degree or strength percentile to define "rich" nodes.
        plot (bool): If True, plot the rich-club subgraph.

    Returns:
        rich_nodes (list): List of node indices in the rich club.
        G (networkx.Graph or DiGraph): Full graph object.
        subG (networkx.Graph or DiGraph): Subgraph of rich club.
    """
    # Create directed weighted graph
    G = nx.from_numpy_array(connectivity_matrix, create_using=nx.DiGraph)

    # Compute strength of each node (sum of in+out weights)
    strength = {n: sum([d['weight'] for u, v, d in G.edges(n, data=True)]) +
                   sum([d['weight'] for u, v, d in G.in_edges(n, data=True)])
                for n in G.nodes}

    # Determine rich nodes by strength
    threshold = np.percentile(list(strength.values()), percentile_threshold)
    rich_nodes = [n for n, s in strength.items() if s >= threshold]

    # Subgraph of rich nodes
    subG = G.subgraph(rich_nodes)

    # Compute rich-club coefficient (optionally normalized)
    rc_density = nx.density(subG)
    print(f"Rich-club of {len(rich_nodes)} nodes has density: {rc_density:.3f}")

    if plot:
        plt.figure(figsize=(6, 6))
        pos = nx.spring_layout(subG)
        nx.draw_networkx(subG, pos, with_labels=True, node_color='orange', edge_color='gray')
        plt.title("Rich Club Subgraph")
        plt.show()

    return rich_nodes, G, subG


In [ ]:
# Suppose conn_matrix is your N x N numpy array
rich_nodes, G, subG = detect_rich_club(adj1.values)


In [ ]:
# Group by the unique names of rows and columns and sample 10% of each group
adj1_sample= adj1.groupby(axis=0, level=0)
adj1_sample = adj1_sample.apply(lambda x: x.sample(frac=0.1, random_state=1))
adj1_sample = adj1_sample.groupby(axis=1, level=0).apply(lambda x: x.sample(frac=0.1, axis=1, random_state=1))

# Print the shape of the sampled matrix
print(adj1_sample.shape)
adj1_sample = adj1_sample.sort_index().sort_index(axis=1)
#sort col
adj1_sample = adj1_sample.sort_index(axis=1)
adj1_sample.reset_index(drop=True, level=0, inplace=True)

# Reset the columns to remove additional column index names
adj1_sample.columns = adj1_sample.columns.droplevel(0)


In [ ]:
#make a dataframe wheere cols and rows with sam ename are grouped and mean given
adj1_df_types = adj1.groupby(axis=0, level=0).mean()
adj1_df_type = adj1_df_types.groupby(axis=1, level=0).mean()


In [ ]:
adj1_df_type

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
#
# Assuming adj1_df_types_sample is already created
# Get log2 of the matrix
log_matrix = np.log2((1+adj1_sample))
log_matrix.replace([np.inf, -np.inf], 0, inplace=True)

# Normalize the matrix (if required, otherwise skip this step)
norm_matrix = log_matrix # Here it's simply assigned directly
norm_matrix.drop('td3', axis=0, inplace=True)
norm_matrix.drop('td3', axis=1, inplace=True)
# Set the colormap to start from the first non-zero value
cmap = 'coolwarm'

# Plot the heatmap
 # Adjust the figure size
#remove col and rows with tn3
plt.figure(figsize=(10, 8))  # Adjust the figure size
# Create the heatmap
heatmap = sns.heatmap(data=norm_matrix, cmap=cmap,  cbar_kws={'label': 'Log2 Intensity'})

#plt.title('Heatmap of Matrix Connections')  # Add title
plt.xlabel('Receiving')  # Add x-axis label
plt.ylabel('Sending')  # Add y-axis label
# Set font size for x-axis and y-axis labels
#heatmap.set_xticklabels(heatmap.get_xticklabels(), fontsize=13, rotation=45, ha='right')  # Adjust font size and rotation for x-axis labels
#heatmap.set_yticklabels(heatmap.get_yticklabels(), fontsize=13)  # Adjust font size for y-axis labels


#binarize the norm_matrix = np.where(norm_matrix > 0, 1, 0)
#norm_matrix_bin = np.where(norm_matrix > 0, 1, 0)

In [ ]:
en_subset = norm_matrix.T[norm_matrix.T.index.str.startswith('en')]

# Plot the heatmap
plt.figure(figsize=(12, 7))
sns.heatmap(data=en_subset.T, cmap=cmap, fmt=".1f", cbar_kws={'label': 'Log2 Intensity'})
#plt.title('Heatmap of Matrix Connections (Subset: Rows starting with "en")')
plt.xlabel('Receiving')
plt.ylabel('Sending')
plt.tight_layout()
plt.show()

In [ ]:
en_subset = norm_matrix[norm_matrix.index.str.startswith('en')]

# Plot the heatmap
plt.figure(figsize=(12, 7))
sns.heatmap(data=en_subset, cmap=cmap, fmt=".1f", cbar_kws={'label': 'Log2 Intensity'})
#plt.title('Heatmap of Matrix Connections (Subset: Rows starting with "en")')
plt.xlabel('Receiving')
plt.ylabel('Sending')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list

# Normalize the matrix
log_matrix = np.log2(1 + adj1_df_type)
log_matrix.replace([np.inf, -np.inf], 0, inplace=True)


log_matrix.drop('td3', axis=0, inplace=True, errors='ignore')
log_matrix.drop('td3', axis=1, inplace=True, errors='ignore')


In [ ]:
plt.figure(figsize=(20, 20))

# Create row colors DataFrame
row_colors_df = pd.DataFrame({'Color': custom_colors[:len(cell_order)]}, index=cell_order)

# Convert row_colors_df to a dictionary
row_colors_dic = row_colors_df['Color'].to_dict()
clustermap = sns.clustermap(
    data=log_matrix,
    row_cluster=True,
    col_cluster=True,
    cmap='coolwarm',
    fmt=".1f",

    #cbar_kws={'label': 'Sum of Log2 Intensity'},
    # Get the labels of the rows and columns
    row_colors= [row_colors_dic.get(index, '#FFFFFF') for index in log_matrix.index],
    col_colors=[row_colors_dic.get(index, '#FFFFFF') for index in log_matrix.columns],
    yticklabels=True,
    xticklabels=True,
    dendrogram_ratio=(0.1, 0.1),




    #dendrogram_ratio=(.2, .2),  # Adjust the size of the dendrogram
    #cbar_pos=(1, .2, .01, .4)  # Adjust the color bar position
)

# Customize the labels
clustermap.ax_heatmap.set_xlabel('Receiving', fontsize=12)
clustermap.ax_heatmap.set_ylabel('Sending', fontsize=12)

# Rotate the x-axis labels for better visibility

# Adjust the layout to make space for the labels
#plt.tight_layout()

# Display the plot

# Customize tick labels font size
plt.setp(clustermap.ax_heatmap.yaxis.get_majorticklabels(), fontsize=12)
plt.setp(clustermap.ax_heatmap.xaxis.get_majorticklabels(), fontsize=12)

# Show the plot
plt.show()




In [ ]:

#make a color vecotr of the order of the columns and then rows of norm marix
colors_bi = [row_colors_dic.get(index, '#FFFFFF') for index in norm_matrix_sum.index]
#append for cols
colors_c = [row_colors_dic.get(index, '#FFFFFF') for index in norm_matrix_sum.T.index]
#append both
colors_all = colors_bi + colors_c

In [ ]:
#save norm_matrix_sum as csv
log_matrix.to_csv('/content/drive/MyDrive/Yuste_Paper_Figures/summed_celltypes.csv')

In [ ]:
#load norm matrix
norm_matrix_sum = pd.read_csv('/content/drive/MyDrive/Yuste_Paper_Figures/summed_celltypes.csv', index_col=0)

In [ ]:
pip install pystruct


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Example adjacency matrix with row and column names
adj_matrix = np.array(norm_matrix_sum)
node_labels = norm_matrix_sum.index

# Create directed graph from adjacency matrix
G = nx.from_numpy_array(adj_matrix, create_using=nx.DiGraph)
G = nx.relabel_nodes(G, {i: label for i, label in enumerate(node_labels)})

# Node size based on degree
node_sizes = [50 * (G.in_degree(n) + G.out_degree(n)) for n in G.nodes()]

# Node color based on prefix: 'en' = orange, 'ec' = light purple
node_color_map = {
    node: '#f28e2b' if node.startswith('en') else '#c5b0d5'
    for node in G.nodes()
}
node_colors = [node_color_map[n] for n in G.nodes()]

# Edge weights for color/width
edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
norm = plt.Normalize(min(edge_weights), max(edge_weights))

# Edge colormap
cmap = plt.cm.get_cmap('coolwarm')

# Draw graph
plt.figure(figsize=(10, 8))
#pos = nx.spring_layout(G, seed=42)
pos = nx.shell_layout(G, nlist=[['en1', 'en2', "en3"], ['ec1A',"ec1B", 'ec2', 'ec3A', "ec3B","ec3C", "ec4", "ec5"]])


# Draw nodes
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors)


nx.draw_networkx_edges(
    G, pos,
    edge_color=edge_weights,

    edge_cmap=plt.cm.Greys,  # ⬅️ Light gray to black
    edge_vmin=min(edge_weights),
    edge_vmax=max(edge_weights),
    #width=[0.5+ 4 * norm(w) for w in edge_weights],  # ⬅️ Scaled width
    arrowstyle='-|>',
    arrowsize=20,
    connectionstyle='arc3,rad=0.1'
)

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=12, font_color='black')

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
#plt.colorbar(sm, label='Connection Strength')

plt.title("Directed Graph from Adjacency Matrix (Orange = en, Purple = ec)")
plt.axis('off')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
pos = nx.kamada_kawai_layout(G)

# Draw nodes with black border
nx.draw_networkx_nodes(
    G, pos,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors='black',
    linewidths=1.5,
    ax=ax
)

# Draw edges with bigger arrows and widths
nx.draw_networkx_edges(
    G, pos,
    edge_color=edge_weights,
    edge_cmap=plt.cm.viridis,
    edge_vmin=min(edge_weights),
    edge_vmax=max(edge_weights),
    width=[1 + 2 * norm(w) for w in edge_weights],
    arrowstyle='-|>',
    arrowsize=10,
    connectionstyle='arc3,rad=0.12',
    ax=ax
)

nx.draw_networkx_labels(G, pos, font_size=12, font_color='black', ax=ax)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=norm)
sm.set_array([])
fig.colorbar(sm, ax=ax, label='Connection Strength')

ax.set_title("Directed Graph showing dense Neuropeptide Connectome(Orange = en, Purple = ec)")
ax.axis('off')
plt.show()


In [ ]:
# now subsetting strognest edges
# 1. Compute threshold
threshold = np.percentile(edge_weights, 60)

# 2. Filter edges
strong_edges = [(u, v) for u, v in G.edges() if G[u][v]['weight'] >= threshold]
G_strong = G.edge_subgraph(strong_edges).copy()

# 3. Get edge weights and node attributes
edge_weights_strong = [G_strong[u][v]['weight'] for u, v in G_strong.edges()]
norm_strong = plt.Normalize(min(edge_weights_strong), max(edge_weights_strong))

node_sizes_strong = [50 * (G_strong.in_degree(n) + G_strong.out_degree(n)) for n in G_strong.nodes()]
node_colors_strong = ['#f28e2b' if n.startswith('en') else '#c5b0d5' for n in G_strong.nodes()]

# 4. Draw
fig, ax = plt.subplots(figsize=(10, 8))
pos = nx.kamada_kawai_layout(G_strong)

nx.draw_networkx_nodes(G_strong, pos, node_size=node_sizes_strong, node_color=node_colors_strong,
                       edgecolors='black', linewidths=1.5, ax=ax)
nx.draw_networkx_labels(G_strong, pos, font_size=12, font_color='black', ax=ax)

sm = plt.cm.ScalarMappable(cmap=plt.cm.viridis, norm=norm_strong)
sm.set_array([])
fig.colorbar(sm, ax=ax, label='Connection Strength')

ax.set_title("Directed Graph showing dense core Neuropeptide Connectome(Orange = en, Purple = ec)")
nx.draw_networkx_edges(
    G_strong, pos,
    edge_color=edge_weights_strong,
    edge_cmap=plt.cm.viridis,  # or magma, inferno, etc.
    edge_vmin=min(edge_weights_strong),
    edge_vmax=max(edge_weights_strong),
    width=[1 + 2 * norm_strong(w) for w in edge_weights_strong],
    arrowstyle='-|>', arrowsize=15,
    connectionstyle='arc3,rad=0.12',
    ax=ax
)


ax.axis('off')
plt.show()


In [ ]:
# === Filter edges: en → ec ===
edges_en_to_ec = [(u, v) for u, v in G.edges() if u.startswith('en') and v.startswith('ec')]
G_en_to_ec = G.edge_subgraph(edges_en_to_ec).copy()

# === Filter edges: ec → en ===
edges_ec_to_en = [(u, v) for u, v in G.edges() if u.startswith('ec') and v.startswith('en')]
G_ec_to_en = G.edge_subgraph(edges_ec_to_en).copy()

# === Define a common draw function ===
def draw_subgraph(G_sub, title, cmap=plt.cm.viridis):
  #subset to strongest coonections

    edge_weights = [G_sub[u][v]['weight'] for u, v in G_sub.edges()]



    threshold = np.percentile(edge_weights, 60)
    strong_edges = [(u, v) for u, v in G_sub.edges() if G_sub[u][v]['weight'] >= threshold]
    G = G_sub.edge_subgraph(strong_edges).copy()
    edge_weights_sub = [G[u][v]['weight'] for u, v in G.edges()]
    fig, ax = plt.subplots(figsize=(10, 8))
    pos = nx.kamada_kawai_layout(G)
    norm = plt.Normalize(min(edge_weights), max(edge_weights))
    pos = nx.shell_layout(G, nlist=[['en1', 'en2', "en3"], ['ec1A',"ec1B", 'ec2', 'ec3A', "ec3B","ec3C", "ec4", "ec5"]])
    node_sizes = [50 * (G.in_degree(n) + G.out_degree(n)) for n in G.nodes()]
    node_colors = ['#f28e2b' if n.startswith('en') else '#c5b0d5' for n in G.nodes()]

     #sum 1 to positions
    pos_lables = {node: (x+0.04, y + 0.05) for node, (x, y) in pos.items()}

    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors,
                           edgecolors='black', linewidths=1.5, ax=ax)

    nx.draw_networkx_edges(
        G, pos,
        edge_color=edge_weights_sub,
        edge_cmap=cmap,
        edge_vmin=min(edge_weights_sub),
        edge_vmax=max(edge_weights_sub),
        width=[1 + 2 * norm(w) for w in edge_weights_sub],
        arrowstyle='-|>',
        arrowsize=10,
        connectionstyle='arc3,rad=0.12',
        ax=ax
    )

    nx.draw_networkx_labels(G, pos_lables, font_size=12, font_color='black', ax=ax)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, label='Connection Strength')

    ax.set_title(title)
    ax.axis('off')
    plt.show()

# === Draw both directional graphs ===
draw_subgraph(G_en_to_ec, "Connections from Endoderm to Ectoderm")
draw_subgraph(G_ec_to_en, "Connections from Ectoderm to Endoderm")


In [ ]:
import seaborn as sns
import pandas as pd

# Create adjacency matrix: en rows x ec columns
en_nodes = [n for n in G.nodes if n.startswith('en')]
ec_nodes = [n for n in G.nodes if n.startswith('ec')]
conn_matrix = pd.DataFrame(0, index=en_nodes, columns=ec_nodes)

for u, v in G.edges():
    if u in en_nodes and v in ec_nodes:
        conn_matrix.loc[u, v] = G[u][v]['weight']

sns.heatmap(conn_matrix, cmap='viridis', annot=True, linewidths=0.5)
plt.title("Connectivity from Endoderm to Ectoderm")
plt.xlabel("Recieving Ectoderm nodes")
plt.ylabel("Sending Endoderm nodes")
plt.show()
import seaborn as sns
import pandas as pd

# Define nodes
en_nodes = [n for n in G.nodes if n.startswith('en')]
ec_nodes = [n for n in G.nodes if n.startswith('ec')]

# Create adjacency matrix: ec rows x en columns
conn_matrix_inv = pd.DataFrame(0, index=ec_nodes, columns=en_nodes)

# Fill with edge weights from ec to en
for u, v in G.edges():
    if u in ec_nodes and v in en_nodes:
        conn_matrix_inv.loc[u, v] = G[u][v]['weight']

# Plot heatmap
sns.heatmap(conn_matrix_inv, cmap='viridis', annot=True, linewidths=0.5)
plt.title("Connectivity from Ectoderm to Endoderm")
plt.xlabel("Recieving Endoderm nodes")
plt.ylabel("Sending Ectoderm nodes")
plt.show()



In [ ]:
pos = nx.bipartite_layout(G_en_to_ec, nodes=en_nodes)
nx.draw(G_en_to_ec, pos, with_labels=True, edge_color=edge_weights, ...)


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
custom_colors = ["#00FF00", "#ff7f0e", "#2ca02c", "#FF0000", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#FFFF00", "#00FFFF", "#a33a22"]

# Define the order of cell names
cell_order = ['en2', 'ec2', 'ec1A', 'ec4', 'ec3C', 'ec3A', 'en3', 'en1', 'ec5', 'ec3B', 'ec1B']

# Create row colors DataFrame
row_colors_df = pd.DataFrame({'Color': custom_colors[:len(cell_order)]}, index=cell_order)

# Convert row_colors_df to a dictionary
row_colors_dic = row_colors_df['Color'].to_dict()
# Example adjacency matrix with row and column names
adj_matrix = np.array(norm_matrix_sum )

# Replace these with your actual row/column names
node_labels = norm_matrix_sum.index

# Create a directed graph from the adjacency matrix
G = nx.from_numpy_array(adj_matrix, create_using=nx.DiGraph)

# Map node indices to labels
G = nx.relabel_nodes(G, {i: label for i, label in enumerate(node_labels)})

# Calculate node sizes based on degree (sum of in-degree and out-degree)
node_sizes = [50 * (G.in_degree(n) + G.out_degree(n)) for n in G.nodes()]
node_colors = colors_all#['red', 'blue', 'green', 'purple', 'orange', 'brown', 'pink', 'gray', 'olive', 'cyan', 'magenta','red', 'blue', 'green', 'purple', 'orange', 'brown', 'pink', 'gray', 'olive', 'cyan', 'magenta']
node_color_map = {node: color for node, color in zip(G.nodes(), node_colors)}

# Extract edge weights for edge colors
edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
node_color=[node_color_map[node] for node in G.nodes()]
# Normalize edge weights for colormap
norm = plt.Normalize(min(edge_weights), max(edge_weights))

# Define a colormap from blue to red
cmap = plt.cm.get_cmap('coolwarm')

# Draw the graph
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G)  # You can use other layouts as well

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_color)

# Draw edges with colormap
edges = nx.draw_networkx_edges(
    G, pos, arrowstyle='-|>', arrowsize=20, edge_color=edge_weights, edge_cmap=cmap, edge_vmin=min(edge_weights), edge_vmax=max(edge_weights), width=2, connectionstyle='arc3,rad=0.1'
)

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=12, font_color='black')

# Add colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
#plt.colorbar(sm, label='Connection Strength')

plt.title("Directed Graph from Adjacency Matrix")
plt.show()




In [ ]:
# Filter the graph to only include the top 20% strongest connections
threshold = np.percentile(edge_weights, 70)
edges_to_keep = [(u, v) for u, v, w in G.edges(data=True) if w['weight'] >= threshold]

# Create a subgraph with only the strongest edges
H = G.edge_subgraph(edges_to_keep).copy()

# Add isolated nodes to the subgraph
isolated_nodes = set(G.nodes()) - set(H.nodes())
for node in isolated_nodes:
    H.add_node(node)

# Calculate node sizes for the subgraph
node_sizes_H = [60 * (H.in_degree(n) + H.out_degree(n) + 4) for n in H.nodes()]

# Extract edge weights for the subgraph
edge_weights_H = [H[u][v]['weight'] for u, v in H.edges()]

# Normalize edge weights for the subgraph colormap
norm_H = plt.Normalize(min(edge_weights_H), max(edge_weights_H))

# Draw the filtered graph
plt.figure(figsize=(10, 8))
pos_H = nx.spring_layout(G)  # You can use other layouts as well

# Assign a default node color map
#node_color_map = {node: 'lightblue' for node in G.nodes()}
node_color = [node_color_map[node] for node in H.nodes()]
node_color=[node_color_map[node] for node in H.nodes()]
# Draw nodes for the subgraph
nx.draw_networkx_nodes(H, pos_H, node_size=node_sizes_H, node_color=node_color)

# Draw edges for the subgraph with colormap
edges_H = nx.draw_networkx_edges(
    H, pos_H, arrowstyle='-|>', arrowsize=20, edge_color=edge_weights_H, edge_cmap=cmap, edge_vmin=min(edge_weights_H), edge_vmax=max(edge_weights_H), width=2, connectionstyle='arc3,rad=0.1'
)

# Draw labels for the subgraph
nx.draw_networkx_labels(H, pos_H, font_size=12, font_color='black')

# Add colorbar for the subgraph
sm_H = plt.cm.ScalarMappable(cmap=cmap, norm=norm_H)
sm_H.set_array([])
plt.colorbar(sm_H, label='Connection Strength')

plt.title("Filtered Graph with Top 30% Strongest Connections")
plt.show()


In [ ]:
# Filter the graph for edges from 'ec' to 'en'
edges_ec_to_en = [(u, v) for u, v in G.edges() if u.startswith('ec') and v.startswith('en')]
H_ec_to_en = G.edge_subgraph(edges_ec_to_en).copy()

# Add isolated nodes to the subgraph
isolated_nodes = set(G.nodes()) - set(H_ec_to_en.nodes())
for node in isolated_nodes:
    H_ec_to_en.add_node(node)

# Calculate node sizes for the subgraph
node_sizes_H_ec_to_en = [50 * (H_ec_to_en.in_degree(n) + H_ec_to_en.out_degree(n) + 4) for n in H_ec_to_en.nodes()]

# Extract edge weights for the subgraph
edge_weights_H_ec_to_en = [H_ec_to_en[u][v]['weight'] for u, v in H_ec_to_en.edges()]

# Normalize edge weights for the subgraph colormap
norm_H_ec_to_en = plt.Normalize(min(edge_weights_H_ec_to_en), max(edge_weights_H_ec_to_en))

# Draw the filtered graph
plt.figure(figsize=(10, 8))
pos_H_ec_to_en = nx.spring_layout(G) # You can use other layouts as well

# Assign a default node color map
node_color_map = {node: 'lightblue' for node in G.nodes()}
node_color = [node_color_map[node] for node in H_ec_to_en.nodes()]

# Draw nodes for the subgraph
nx.draw_networkx_nodes(H_ec_to_en, pos_H_ec_to_en, node_size=node_sizes_H_ec_to_en, node_color=['#fdb462' if node.startswith('ec') else '#66c2a5' for node in G.nodes()])
# Draw nodes
#nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=['pink' if node.startswith('ec') else 'lightgreen' for node in G.nodes()])

# Draw edges for the subgraph with colormap
edges_H_ec_to_en = nx.draw_networkx_edges(
    H_ec_to_en, pos_H_ec_to_en, arrowstyle='-|>', arrowsize=20, edge_color=edge_weights_H_ec_to_en, edge_cmap=cmap, edge_vmin=min(edge_weights_H_ec_to_en), edge_vmax=max(edge_weights_H_ec_to_en), width=2, connectionstyle='arc3,rad=0.1'
)

# Draw labels for the subgraph
nx.draw_networkx_labels(H_ec_to_en, pos_H_ec_to_en, font_size=12, font_color='black')

# Add colorbar for the subgraph
sm_H_ec_to_en = plt.cm.ScalarMappable(cmap=cmap, norm=norm_H_ec_to_en)
sm_H_ec_to_en.set_array([])
#plt.colorbar(sm_H_ec_to_en, label='Connection Strength')

plt.title("Filtered Graph with Connections from 'ec' to 'en'")
plt.show()

# Filter the graph for edges from 'en' to 'ec'
edges_en_to_ec = [(u, v) for u, v in G.edges() if u.startswith('en') and v.startswith('ec')]
H_en_to_ec = G.edge_subgraph(edges_en_to_ec).copy()

# Add isolated nodes to the subgraph
isolated_nodes = set(G.nodes()) - set(H_en_to_ec.nodes())
for node in isolated_nodes:
    H_en_to_ec.add_node(node)

# Calculate node sizes for the subgraph
node_sizes_H_en_to_ec = [50 * (H_en_to_ec.in_degree(n) + H_en_to_ec.out_degree(n) + 4) for n in H_en_to_ec.nodes()]

# Extract edge weights for the subgraph
edge_weights_H_en_to_ec = [H_en_to_ec[u][v]['weight'] for u, v in H_en_to_ec.edges()]

# Normalize edge weights for the subgraph colormap
norm_H_en_to_ec = plt.Normalize(min(edge_weights_H_en_to_ec), max(edge_weights_H_en_to_ec))

# Draw the filtered graph
plt.figure(figsize=(10, 8))
pos_H_en_to_ec = nx.spring_layout(G)  # You can use other layouts as well

# Draw nodes for the subgraph
nx.draw_networkx_nodes(H_en_to_ec, pos_H_en_to_ec, node_size=node_sizes_H_en_to_ec, node_color=['mistyrose' if node.startswith('ec') else 'lightblue' for node in G.nodes()])
# Draw nodes
#x.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=['pink' if node.startswith('ec') else 'lightgreen' for node in G.nodes()])

# Draw edges for the subgraph with colormap
edges_H_en_to_ec = nx.draw_networkx_edges(
    H_en_to_ec, pos_H_en_to_ec, arrowstyle='-|>', arrowsize=20, edge_color=edge_weights_H_en_to_ec, edge_cmap=cmap, edge_vmin=min(edge_weights_H_en_to_ec), edge_vmax=max(edge_weights_H_en_to_ec), width=2, connectionstyle='arc3,rad=0.1'
)

# Draw labels for the subgraph
nx.draw_networkx_labels(H_en_to_ec, pos_H_en_to_ec, font_size=12, font_color='black')

# Add colorbar for the subgraph
sm_H_en_to_ec = plt.cm.ScalarMappable(cmap=cmap, norm=norm_H_en_to_ec)
sm_H_en_to_ec.set_array([])
#plt.colorbar(sm_H_en_to_ec, label='Connection Strength')

plt.title("Filtered Graph with Connections from 'en' to 'ec'")
plt.show()

# Now, filter to include only the top 20% strongest connections for each subgraph
threshold_ec_to_en = np.percentile(edge_weights_H_ec_to_en, 70)
strong_edges_ec_to_en = [(u, v) for u, v, w in H_ec_to_en.edges(data=True) if w['weight'] >= threshold_ec_to_en]
H_strong_ec_to_en = H_ec_to_en.edge_subgraph(strong_edges_ec_to_en).copy()

# Add isolated nodes to the subgraph
isolated_nodes = set(H_ec_to_en.nodes()) - set(H_strong_ec_to_en.nodes())
for node in isolated_nodes:
    H_strong_ec_to_en.add_node(node)

# Calculate node sizes for the subgraph
node_sizes_H_strong_ec_to_en = [50 * (H_strong_ec_to_en.in_degree(n) + H_strong_ec_to_en.out_degree(n) + 4) for n in H_strong_ec_to_en.nodes()]

# Extract edge weights for the subgraph
edge_weights_H_strong_ec_to_en = [H_strong_ec_to_en[u][v]['weight'] for u, v in H_strong_ec_to_en.edges()]

# Normalize edge weights for the subgraph colormap
norm_H_strong_ec_to_en = plt.Normalize(min(edge_weights_H_strong_ec_to_en), max(edge_weights_H_strong_ec_to_en))

# Draw the filtered graph
plt.figure(figsize=(10, 8))
pos_H_strong_ec_to_en = nx.spring_layout(G) # You can use other layouts as well

# Draw nodes for the subgraph
nx.draw_networkx_nodes(H_strong_ec_to_en, pos_H_strong_ec_to_en, node_size=node_sizes_H_strong_ec_to_en, node_color=['mistyrose' if node.startswith('ec') else 'lightblue' for node in G.nodes()])

edges_H_strong_ec_to_en = nx.draw_networkx_edges(
    H_strong_ec_to_en, pos_H_strong_ec_to_en, arrowstyle='-|>', arrowsize=20, edge_color=edge_weights_H_strong_ec_to_en, edge_cmap=cmap, edge_vmin=min(edge_weights_H_strong_ec_to_en), edge_vmax=max(edge_weights_H_strong_ec_to_en), width=2, connectionstyle='arc3,rad=0.1'
)

# Draw labels for the subgraph
nx.draw_networkx_labels(H_strong_ec_to_en, pos_H_strong_ec_to_en, font_size=12, font_color='black')

# Add colorbar for the subgraph
sm_H_strong_ec_to_en = plt.cm.ScalarMappable(cmap=cmap, norm=norm_H_strong_ec_to_en)
sm_H_strong_ec_to_en.set_array([])
#plt.colorbar(sm_H_strong_ec_to_en, label='Connection Strength')

plt.title("Filtered Graph with Top 30% Strongest Connections from 'ec' to 'en'")
plt.show()

# Now, filter to include only the top 20% strongest connections for edges from 'en' to 'ec'
threshold_en_to_ec = np.percentile(edge_weights_H_en_to_ec, 70)
strong_edges_en_to_ec = [(u, v) for u, v, w in H_en_to_ec.edges(data=True) if w['weight'] >= threshold_en_to_ec]
H_strong_en_to_ec = H_en_to_ec.edge_subgraph(strong_edges_en_to_ec).copy()

# Add isolated nodes to the subgraph
isolated_nodes = set(H_en_to_ec.nodes()) - set(H_strong_en_to_ec.nodes())
for node in isolated_nodes:
    H_strong_en_to_ec.add_node(node)

# Calculate node sizes for the subgraph
node_sizes_H_strong_en_to_ec = [50 * (H_strong_en_to_ec.in_degree(n) + H_strong_en_to_ec.out_degree(n) + 4) for n in H_strong_en_to_ec.nodes()]

# Extract edge weights for the subgraph
edge_weights_H_strong_en_to_ec = [H_strong_en_to_ec[u][v]['weight'] for u, v in H_strong_en_to_ec.edges()]

# Normalize edge weights for the subgraph colormap
norm_H_strong_en_to_ec = plt.Normalize(min(edge_weights_H_strong_en_to_ec), max(edge_weights_H_strong_en_to_ec))

# Draw the filtered graph
plt.figure(figsize=(10, 8))
pos_H_strong_en_to_ec = nx.spring_layout(G)  # You can use other layouts as well

# Draw nodes for the subgraph
nx.draw_networkx_nodes(H_strong_en_to_ec, pos_H_strong_en_to_ec, node_size=node_sizes_H_strong_en_to_ec, node_color=['mistyrose' if node.startswith('ec') else 'lightblue' for node in G.nodes()])
# Draw nodes
#nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=['pink' if node.startswith('ec') else 'lightgreen' for node in G.nodes()])

# Draw edges for the subgraph with colormap
edges_H_strong_en_to_ec = nx.draw_networkx_edges(
    H_strong_en_to_ec, pos_H_strong_en_to_ec, arrowstyle='-|>', arrowsize=20, edge_color=edge_weights_H_strong_en_to_ec, edge_cmap=cmap, edge_vmin=min(edge_weights_H_strong_en_to_ec), edge_vmax=max(edge_weights_H_strong_en_to_ec), width=2, connectionstyle='arc3,rad=0.1'
)

# Draw labels for the subgraph
nx.draw_networkx_labels(H_strong_en_to_ec, pos_H_strong_en_to_ec, font_size=12, font_color='black')

# Add colorbar for the subgraph
sm_H_strong_en_to_ec = plt.cm.ScalarMappable(cmap=cmap, norm=norm_H_strong_en_to_ec)
sm_H_strong_en_to_ec.set_array([])
#plt.colorbar(sm_H_strong_en_to_ec, label='Connection Strength')

plt.title("Filtered Graph with Top 30% Strongest Connections from 'en' to 'ec'")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# --- Custom colormap: warm gold -> muted magenta -> deep purple ---
custom_cmap = LinearSegmentedColormap.from_list(
    "custom_edge_colormap",
    ['#fdbb84', '#e34a33', '#6a3d9a']  # gold → orange-red → purple
)

# --- Draw a subgraph with color and width mapped to edge weights ---
def draw_weighted_directed_graph(H, title):
    # Calculate layout (shared layout across all graphs for consistent comparison)
    pos = nx.spring_layout(H, seed=42)

    # Node sizes based on degree
    node_sizes = [50 * (H.in_degree(n) + H.out_degree(n) + 4) for n in H.nodes()]

    # Assign node colors
    node_colors = ['#e8d6ff' if node.startswith('ec') else '#b6e3c1' for node in H.nodes()]  # soft lilac & mint

    # Extract edge weights
    edge_weights = [H[u][v]['weight'] for u, v in H.edges()]
    norm = plt.Normalize(min(edge_weights), max(edge_weights))

    # Scale edge widths (more weight → thicker)
    edge_widths = [1 + 3 * norm(w) for w in edge_weights]  # from 1 to 4 width

    # Plot
    plt.figure(figsize=(10, 8))
    nx.draw_networkx_nodes(H, pos, node_size=node_sizes, node_color=node_colors)
    nx.draw_networkx_labels(H, pos, font_size=11, font_color='black')

    # Draw weighted and colored edges
    edges = nx.draw_networkx_edges(
        H, pos,
        edge_color=edge_weights,
        edge_cmap=custom_cmap,
        edge_vmin=min(edge_weights),
        edge_vmax=max(edge_weights),
        width=edge_widths,
        arrowstyle='-|>',
        arrowsize=20,
        connectionstyle='arc3,rad=0.12'
    )

    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, label='Connection Strength')

    plt.title(title)
    plt.axis('off')
    plt.show()


In [ ]:
draw_weighted_directed_graph(H_strong_en_to_ec, "Top 30% Strongest Connections from 'en' to 'ec'")
draw_weighted_directed_graph(H_strong_ec_to_en, "Top 30% Strongest Connections from 'ec' to 'en'")


In [ ]:
# Filter the graph for edges from 'ec' to 'en'
edges_ec_to_en = [(u, v) for u, v in G.edges() if u.startswith('en1')]
H_ec_to_en = G.edge_subgraph(edges_ec_to_en).copy()

# Add isolated nodes to the subgraph
isolated_nodes = set(G.nodes()) - set(H_ec_to_en.nodes())
for node in isolated_nodes:
    H_ec_to_en.add_node(node)

# Calculate node sizes for the subgraph
node_sizes_H_ec_to_en = [50 * (H_ec_to_en.in_degree(n) + H_ec_to_en.out_degree(n) + 4) for n in H_ec_to_en.nodes()]

# Extract edge weights for the subgraph
edge_weights_H_ec_to_en = [H_ec_to_en[u][v]['weight'] for u, v in H_ec_to_en.edges()]

# Normalize edge weights for the subgraph colormap
norm_H_ec_to_en = plt.Normalize(min(edge_weights_H_ec_to_en), max(edge_weights_H_ec_to_en))

# Draw the filtered graph
plt.figure(figsize=(10, 8))
pos_H_ec_to_en = nx.spring_layout(G) # You can use other layouts as well

# Assign a default node color map
node_color_map = {node: 'lightblue' for node in G.nodes()}
node_color = [node_color_map[node] for node in H_ec_to_en.nodes()]

# Draw nodes for the subgraph
nx.draw_networkx_nodes(H_ec_to_en, pos_H_ec_to_en, node_size=node_sizes_H_ec_to_en, node_color=['mistyrose' if node.startswith('ec') else 'lightblue' for node in G.nodes()])
# Draw nodes
#nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=['pink' if node.startswith('ec') else 'lightgreen' for node in G.nodes()])

# Draw edges for the subgraph with colormap
edges_H_ec_to_en = nx.draw_networkx_edges(
    H_ec_to_en, pos_H_ec_to_en, arrowstyle='-|>', arrowsize=20, edge_color=edge_weights_H_ec_to_en, edge_cmap=cmap, edge_vmin=min(edge_weights_H_ec_to_en), edge_vmax=max(edge_weights_H_ec_to_en), width=2, connectionstyle='arc3,rad=0.1'
)

# Draw labels for the subgraph
nx.draw_networkx_labels(H_ec_to_en, pos_H_ec_to_en, font_size=12, font_color='black')

# Add colorbar for the subgraph
sm_H_ec_to_en = plt.cm.ScalarMappable(cmap=cmap, norm=norm_H_ec_to_en)
sm_H_ec_to_en.set_array([])
plt.colorbar(sm_H_ec_to_en, label='Connection Strength')

plt.title("Filtered Graph with Connections from 'ec' to 'en'")
plt.show()

# Filter the graph for edges from 'en' to 'ec'
edges_en_to_ec = [(u, v) for u, v in G.edges() if u.startswith('en') and v.startswith('ec')]
H_en_to_ec = G.edge_subgraph(edges_en_to_ec).copy()

# Add isolated nodes to the subgraph
isolated_nodes = set(G.nodes()) - set(H_en_to_ec.nodes())
for node in isolated_nodes:
    H_en_to_ec.add_node(node)

# Calculate node sizes for the subgraph
node_sizes_H_en_to_ec = [50 * (H_en_to_ec.in_degree(n) + H_en_to_ec.out_degree(n) + 4) for n in H_en_to_ec.nodes()]

# Extract edge weights for the subgraph
edge_weights_H_en_to_ec = [H_en_to_ec[u][v]['weight'] for u, v in H_en_to_ec.edges()]

# Normalize edge weights for the subgraph colormap
norm_H_en_to_ec = plt.Normalize(min(edge_weights_H_en_to_ec), max(edge_weights_H_en_to_ec))

# Draw the filtered graph
plt.figure(figsize=(10, 8))
pos_H_en_to_ec = nx.spring_layout(G)  # You can use other layouts as well

# Draw nodes for the subgraph
nx.draw_networkx_nodes(H_en_to_ec, pos_H_en_to_ec, node_size=node_sizes_H_en_to_ec, node_color=['mistyrose' if node.startswith('ec') else 'lightblue' for node in G.nodes()])
# Draw nodes
#x.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=['pink' if node.startswith('ec') else 'lightgreen' for node in G.nodes()])

# Draw edges for the subgraph with colormap
edges_H_en_to_ec = nx.draw_networkx_edges(
    H_en_to_ec, pos_H_en_to_ec, arrowstyle='-|>', arrowsize=20, edge_color=edge_weights_H_en_to_ec, edge_cmap=cmap, edge_vmin=min(edge_weights_H_en_to_ec), edge_vmax=max(edge_weights_H_en_to_ec), width=2, connectionstyle='arc3,rad=0.1'
)

# Draw labels for the subgraph
nx.draw_networkx_labels(H_en_to_ec, pos_H_en_to_ec, font_size=12, font_color='black')

# Add colorbar for the subgraph
sm_H_en_to_ec = plt.cm.ScalarMappable(cmap=cmap, norm=norm_H_en_to_ec)
sm_H_en_to_ec.set_array([])
plt.colorbar(sm_H_en_to_ec, label='Connection Strength')

plt.title("Filtered Graph with Connections from 'en' to 'ec'")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

# Custom colormap: gold → red → purple
custom_cmap = LinearSegmentedColormap.from_list(
    "custom_cmap", ['lightgrey', 'grey', 'black']
)

def draw_directed_subgraph(G, edge_filter_fn, title):
    # Filter edges and create subgraph
    selected_edges = [edge for edge in G.edges() if edge_filter_fn(edge)]
    H = G.edge_subgraph(selected_edges).copy()

    # Add isolated nodes to retain full context
    for node in set(G.nodes()) - set(H.nodes()):
        H.add_node(node)

    # Node sizes and colors
    node_sizes = [50 * (H.in_degree(n) + H.out_degree(n) + 4) for n in H.nodes()]
    node_colors = ['#e8d6ff' if n.startswith('ec') else 'orange' for n in H.nodes()]


    # Edge weights, normalization, and width scaling
    edge_weights = [H[u][v]['weight'] for u, v in H.edges()]
    norm = plt.Normalize(min(edge_weights), max(edge_weights))
    edge_widths = [1 + 3 * norm(w) for w in edge_weights]

    # Layout fixed by full graph for consistency
    pos = nx.spring_layout(G, seed=42)

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    nx.draw_networkx_nodes(H, pos, node_size=node_sizes, node_color=node_colors, ax=ax)
    nx.draw_networkx_labels(H, pos, font_size=11, font_color='black', ax=ax)
    nx.draw_networkx_edges(
        H, pos,
        edge_color=edge_weights,
        edge_cmap=custom_cmap,
        edge_vmin=min(edge_weights),
        edge_vmax=max(edge_weights),
        width=edge_widths,
        arrowstyle='-|>',
        arrowsize=20,
        connectionstyle='arc3,rad=0.12',
        ax=ax
    )

    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=norm)
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Connection Strength')

    ax.set_title(title)
    ax.axis('off')
    plt.show()

# Define filters
filter_en_to_ec = lambda edge: edge[0].startswith('en') and edge[1].startswith('ec')
filter_ec_to_en1 = lambda edge: edge[0] == 'en1'  # You originally used en1 only

# Draw both directions
draw_directed_subgraph(G, filter_en_to_ec, "Connections from 'en' to 'ec'")
draw_directed_subgraph(G, filter_ec_to_en1, "Connections from 'en1' to any target")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib.colors as mcolors
import matplotlib.font_manager as fm

# Define the font properties
font_prop = fm.FontProperties(fname='arial', size=10)
# Step 1: Create a sample adjacency matrix
# Assuming norm_matrix_sum is already defined
bipartite_df = norm_matrix_sum.copy()

# Rename columns and index with the actual names
bipartite_df.columns = [f' {col}' for col in bipartite_df.columns]
bipartite_df.index = [f'{index} ' for index in bipartite_df.index]


# Step 3: Plot the Bipartite Graph
# Create a bipartite graph
B = nx.Graph()

# Add nodes with the bipartite attribute
rows = bipartite_df.index
cols = bipartite_df.columns
B.add_nodes_from(rows, bipartite=0)  # From nodes
B.add_nodes_from(cols, bipartite=1)  # To nodes

# Add edges with weights
edges = []
for row in rows:
    for col in cols:
        weight = bipartite_df.loc[ row, col]
        if weight > 0:  # Add edge if there's a connection
            edges.append((row, col, weight))
            B.add_edge(row, col, weight=weight)

# Create a layout for our bipartite graph
pos = nx.drawing.layout.bipartite_layout(B, rows)

# Normalize edge weights for colormap
weights = [B[u][v]['weight'] for u, v in B.edges()]
norm = mcolors.Normalize(vmin=min(weights), vmax=max(weights))
cmap = plt.get_cmap('coolwarm')

# Get edge colors based on weights
edge_colors = [cmap(norm(B[u][v]['weight'])) for u, v in B.edges()]

# Define custom node colors
node_colors = colors_all#['red', 'blue', 'green', 'purple', 'orange', 'brown', 'pink', 'gray', 'olive', 'cyan', 'magenta','red', 'blue', 'green', 'purple', 'orange', 'brown', 'pink', 'gray', 'olive', 'cyan', 'magenta']
node_color_map = {node: color for node, color in zip(B.nodes(), node_colors)}

# Increase node size
node_size = 100

# Draw the full graph
plt.figure(figsize=(10, 6))
nx.draw(
    B, pos, with_labels=False,
    node_color=[node_color_map[node] for node in B.nodes()],
    node_size=node_size,
    edge_color=edge_colors,
    width=2,
    edge_cmap=cmap,
    font_size=10,
    font_color='black',
    verticalalignment='top'
)
ax = plt.gca()

# Loop through each node and adjust the label position vertically
for node, (x, y) in pos.items():
    ax.text(x, y+0.02, node, ha='center', va='bottom', fontsize=10, color='black')


plt.title("Bipartite Graph")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
#plt.colorbar(sm, label='Edge Weight')
plt.show()

# Filter to include only the top 50% of connections
threshold = np.percentile(weights, 50)
filtered_edges = [(u, v, w) for u, v, w in edges if w > threshold]

# Create a new bipartite graph for the filtered edges
B_filtered = nx.Graph()
B_filtered.add_nodes_from(rows, bipartite=0)
B_filtered.add_nodes_from(cols, bipartite=1)
B_filtered.add_weighted_edges_from(filtered_edges)

# Get edge colors for the filtered graph
filtered_weights = [B_filtered[u][v]['weight'] for u, v in B_filtered.edges()]
filtered_edge_colors = [cmap(norm(B_filtered[u][v]['weight'])) for u, v in B_filtered.edges()]
#edge_colors = [cmap(norm(B[u][v]['weight'])) for u, v in B.edges()]
#weights = [B[u][v]['weight'] for u, v in B.edges()]
norm = mcolors.Normalize(vmin=min(filtered_weights), vmax=max(filtered_weights))
# Draw the filtered graph
plt.figure(figsize=(10, 6))
# Draw the graph without labels
nx.draw(B_filtered, pos, with_labels=False, node_color=[node_color_map[node] for node in B_filtered.nodes()],
        node_size=node_size, edge_color=edge_colors, edge_cmap=cmap, width=2)

# Get the current axis to access the plotted objects
ax = plt.gca()

# Loop through each node and adjust the label position vertically
for node, (x, y) in pos.items():
    ax.text(x, y+0.02, node, ha='center', va='bottom', fontsize=10, color='black')

plt.title("Bipartite Graph (Top 30% Connections)")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, label='Edge Weight')
plt.show()


In [ ]:
# Filter out edges connecting nodes within the same group
threshold = np.percentile(weights, 50)
filtered_edges = [(u, v, w) for u, v, w in edges if w > threshold]

filtered_edges = [(u, v, w) for u, v, w in filtered_edges  if ("en" in u and "ec" in v) or ("ec" in u and "en" in v)]

# Create a new bipartite graph for the filtered edges
B_filtered = nx.Graph()
B_filtered.add_nodes_from(rows, bipartite=0)
B_filtered.add_nodes_from(cols, bipartite=1)
B_filtered.add_weighted_edges_from(filtered_edges)
norm = mcolors.Normalize(vmin=min(filtered_weights), vmax=max(filtered_weights))
# Get edge colors for the filtered graph
filtered_weights = [B_filtered[u][v]['weight'] for u, v in B_filtered.edges()]
filtered_edge_colors = [cmap(norm(w)) for w in filtered_weights]

# Draw the filtered graph with arrows
plt.figure(figsize=(10, 6))
nx.draw(
    B_filtered, pos, with_labels=True,
    node_color=[node_color_map[node] for node in B_filtered.nodes()],
    node_size=node_size,
    edge_color=filtered_edge_colors,
    width=2,
    edge_cmap=cmap,
    font_size=10,
    font_color='black',
    verticalalignment='bottom',


)

plt.title("Connections between En and Ec")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, label='Edge Weight')
plt.show()


In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

# Set the path to your file
fasta_file = "/content/drive/MyDrive/Yuste_Paper_Figures/NP_YusteLab_ALLPeptides.fasta"

# Define peptide motif patterns
motif_patterns = {
    'GRFamide': r'GRF$',
    'GRYamide': r'GRY$',
    'PRXamide': r'PR.',
    'GLWamide': r'GLW$',
    'KVamide': r'KV$',
    'LTRamide': r'LTR$',
    'GYGYamide': r'GYGY',
    'PALPW': r'PALPW$',
    'INC': r'INC$',
    'DPFD': r'DPFD$',
    'FRamide': r'FR$',
    'PMIE': r'PMIE$',
    'SET': r'SET',

}

# Step 1–2: Read FASTA and parse peptides by gene
peptides = []
with open(fasta_file, 'r') as file:
    current_gene = None
    for line in file:
        line = line.strip()
        if line.startswith('>'):
            match = re.search(r'(HVAEP\d+\.G\d+)', line)
            if match:
                current_gene = match.group(1)
        elif current_gene:
            peptides.append((current_gene, line))

# Step 3–4: Classify each peptide
from collections import defaultdict
import re

# Assuming 'peptides' is a list of (gene, sequence) tuples
# Assuming 'motif_patterns' is a dictionary of {'category': 'regex_pattern'}

gene_motif_counter = defaultdict(Counter)
for gene, seq in peptides:
    found_match = False  # Flag to track if a match is found for the current sequence
    for category, pattern in motif_patterns.items():
        if re.search(pattern, seq):
            gene_motif_counter[gene][category] += 1
            found_match = True
            break  # Exit the inner loop once a match is found
    # If no match was found for the sequence, you might want to handle it, e.g., count as 'no_motif'
    # if not found_match:
    #     gene_motif_counter[gene]['no_motif'] += 1
# Step 5: Convert to DataFrame
records = []
for gene, counts in gene_motif_counter.items():
    for motif, count in counts.items():
        records.append((gene, motif, count))

df = pd.DataFrame(records, columns=["Gene", "Motif", "Count"])
pivot_df = df.pivot(index="Gene", columns="Motif", values="Count").fillna(0)

# Step 6: Plot
pivot_df.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='tab20')
plt.title("Peptide Motif Types per Preprohormone")
plt.xlabel("Preprohormone Gene")
plt.ylabel("Number of Peptides")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.legend(title='Peptide Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


In [ ]:
peptides

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

# ---- SETTINGS ----
# Set the path to your file
fasta_file = "/content/drive/MyDrive/Yuste_Paper_Figures/NP_YusteLab_ALLPeptides.fasta"

# Priority list: first match wins
motif_priority = [
    'GRFamide', 'GRYamide', 'PRXamide', 'GLWamide', 'KVamide',
    'LTRamide', 'GYGYamide', 'PALPW', 'INC', 'DPFD', 'FRamide',
    'PMIE', 'SET', 'Other'
]

motif_patterns = {
    'GRFamide': r'GRF$',
    'GRYamide': r'GRY$',
    'PRXamide': r'PR.',
    'GLWamide': r'GLW$',
    'KVamide': r'KV$',
    'LTRamide': r'LTR$',
    'GYGYamide': r'GYGY',
    'PALPW': r'PALPW$',
    'INC': r'INC$',
    'DPFD': r'DPFD$',
    'FRamide': r'FR$',
    'PMIE': r'PMIE$',
    'SET': r'SET',
    'Other': r'.*'  # fallback
}

# ---- READ FASTA FILE AND PARSE PEPTIDES ----

# ---- READ FASTA FILE AND PARSE PEPTIDES ----
peptides = []
with open(fasta_file, 'r') as file:
    current_gene = None
    current_seq_lines = []
    for line in file:
        line = line.strip()
        if line.startswith('>'):
            # Save the previous peptide
            if current_gene and current_seq_lines:
                peptide_seq = ''.join(current_seq_lines)
                peptides.append((current_gene, peptide_seq))
            # Start a new entry
            match = re.search(r'(HVAEP\d+\.G\d+)', line)
            current_gene = match.group(1) if match else None
            current_seq_lines = []
        elif current_gene:
            current_seq_lines.append(line)
    # Save the last peptide
    if current_gene and current_seq_lines:
        peptide_seq = ''.join(current_seq_lines)
        peptides.append((current_gene, peptide_seq))

# ---- CLASSIFY BY SINGLE MOTIF (FIRST MATCH WINS) ----
seen = set()
gene_motif_counter = defaultdict(Counter)
# ---- CLASSIFY BY SINGLE MOTIF (FIRST MATCH WINS, ALLOW DUPLICATES) ----
gene_motif_counter = defaultdict(Counter)

for gene, seq in peptides:
    for motif in motif_priority:
        if re.search(motif_patterns[motif], seq):
            gene_motif_counter[gene][motif] += 1
            break



# ---- CREATE DATAFRAME ----
data = []
for gene, counter in gene_motif_counter.items():
    for motif, count in counter.items():
        data.append((gene, motif, count))

df = pd.DataFrame(data, columns=["Gene", "Motif", "Count"])
pivot = df.pivot(index="Gene", columns="Motif", values="Count").fillna(0)

# ---- PLOT ----
pivot.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='tab20')
plt.title("Unique Peptide Motif Types per Preprohormone")
plt.xlabel("Preprohormone Gene")
plt.ylabel("Number of Unique Peptides")
plt.xticks(rotation=45, ha='right')
plt.legend(title='Peptide Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


In [ ]:
# ---- CLASSIFY BY SINGLE MOTIF (GROUP BY FAMILY PREFIX like HVAEP9) ----
gene_family_motif_counter = defaultdict(Counter)

for gene, seq in peptides:
    family_prefix = gene.split('.')[1]  # e.g. HVAEP9
    for motif in motif_priority:
        if re.search(motif_patterns[motif], seq):
            gene_family_motif_counter[family_prefix][motif] += 1
            break

# ---- CREATE DATAFRAME ----
data = []
for family, counter in gene_family_motif_counter.items():
    for motif, count in counter.items():
        data.append((family, motif, count))

df = pd.DataFrame(data, columns=["GeneFamily", "Motif", "Count"])
pivot = df.pivot(index="GeneFamily", columns="Motif", values="Count").fillna(0)


In [ ]:
# ---- PLOT ----
pivot.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='tab20')
plt.title("Peptide Motif Types Grouped by Preprohormone Family")
plt.xlabel("Preprohormone Family")
plt.ylabel("Number of Peptides")
plt.xticks(rotation=45, ha='right')
plt.legend(title='Peptide Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


sadly i also need to do some R now